# 🏠 Análisis de Precios de Viviendas — Ames Housing Dataset

**Dataset:** [House Prices - Advanced Regression Techniques](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques/data)  
**Objetivo:** Explorar qué factores determinan el precio de una vivienda y construir un modelo predictivo.

---

## 0. Importación de librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score

import warnings
warnings.filterwarnings('ignore')

# Estilo general
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print('✅ Librerías importadas correctamente')

## 1. Carga del dataset

> 📥 Descargá el dataset desde [Kaggle](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques/data) y colocá `train.csv` en la carpeta `data/raw/`.

In [ ]:
df = pd.read_csv('../data/raw/train.csv')

print(f'📐 Dimensiones: {df.shape[0]} filas × {df.shape[1]} columnas')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().T.style.background_gradient(cmap='Blues')

## 2. Limpieza y transformación de datos

### 2.1 Análisis de valores nulos

In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({'Nulos': missing, 'Porcentaje (%)': missing_pct})
print(missing_df)

# Visualización
fig, ax = plt.subplots(figsize=(14, 6))
colors = ['#e74c3c' if p > 50 else '#e67e22' if p > 20 else '#3498db' for p in missing_pct]
bars = ax.barh(missing_pct.index, missing_pct.values, color=colors)
ax.axvline(50, color='red', linestyle='--', alpha=0.5, label='50% umbral')
ax.axvline(20, color='orange', linestyle='--', alpha=0.5, label='20% umbral')
ax.set_xlabel('Porcentaje de valores nulos (%)')
ax.set_title('Columnas con valores faltantes', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/missing_values.png', dpi=150)
plt.show()

### 2.2 Tratamiento de nulos

In [ ]:
df_clean = df.copy()

# Columnas donde NaN significa 'Sin característica' (ej: sin garage, sin pool)
cols_none = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
             'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
             'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
             'MasVnrType']

for col in cols_none:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna('None')

# Columnas numéricas donde NaN = 0 (sin garage, sin sótano)
cols_zero = ['GarageYrBlt', 'GarageArea', 'GarageCars',
             'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF',
             'BsmtFullBath', 'BsmtHalfBath', 'MasVnrArea']

for col in cols_zero:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna(0)

# LotFrontage: imputar con mediana por vecindario
df_clean['LotFrontage'] = df_clean.groupby('Neighborhood')['LotFrontage'].transform(
    lambda x: x.fillna(x.median())
)

# Electrical: imputar con moda
df_clean['Electrical'] = df_clean['Electrical'].fillna(df_clean['Electrical'].mode()[0])

# Verificación
remaining_nulls = df_clean.isnull().sum().sum()
print(f'✅ Nulos restantes: {remaining_nulls}')
print(f'📊 Dataset limpio: {df_clean.shape[0]} filas × {df_clean.shape[1]} columnas')

### 2.3 Feature Engineering

In [ ]:
# Edad de la casa al momento de venta
df_clean['HouseAge'] = df_clean['YrSold'] - df_clean['YearBuilt']

# Años desde la última remodelación
df_clean['YearsSinceRemod'] = df_clean['YrSold'] - df_clean['YearRemodAdd']

# Superficie total (sótano + planta baja + planta alta)
df_clean['TotalSF'] = df_clean['TotalBsmtSF'] + df_clean['1stFlrSF'] + df_clean['2ndFlrSF']

# Total de baños
df_clean['TotalBaths'] = (df_clean['FullBath'] + 
                           0.5 * df_clean['HalfBath'] + 
                           df_clean['BsmtFullBath'] + 
                           0.5 * df_clean['BsmtHalfBath'])

# ¿Tiene garage?
df_clean['HasGarage'] = (df_clean['GarageArea'] > 0).astype(int)

# ¿Tiene sótano?
df_clean['HasBasement'] = (df_clean['TotalBsmtSF'] > 0).astype(int)

# ¿Tiene pileta?
df_clean['HasPool'] = (df_clean['PoolArea'] > 0).astype(int)

print('✅ Features nuevas creadas:')
nuevas = ['HouseAge', 'YearsSinceRemod', 'TotalSF', 'TotalBaths', 'HasGarage', 'HasBasement', 'HasPool']
print(df_clean[nuevas].describe().T[['mean','min','max']])

### 2.4 Guardado del dataset limpio

In [ ]:
df_clean.to_csv('../data/processed/train_clean.csv', index=False)
print('💾 Dataset limpio guardado en data/processed/train_clean.csv')

---
## 3. Análisis Exploratorio de Datos (EDA)

### 3.1 Distribución del precio de venta

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribución original
sns.histplot(df_clean['SalePrice'], kde=True, ax=axes[0], color='#3498db')
axes[0].set_title('Distribución del Precio de Venta', fontweight='bold')
axes[0].set_xlabel('Precio (USD)')
axes[0].xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# Distribución log-transformada
sns.histplot(np.log1p(df_clean['SalePrice']), kde=True, ax=axes[1], color='#e74c3c')
axes[1].set_title('Distribución Log(Precio de Venta)', fontweight='bold')
axes[1].set_xlabel('log(Precio)')

for ax in axes:
    ax.set_ylabel('Frecuencia')

plt.suptitle('Precio de Venta: Original vs Log-transformado', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../reports/figures/price_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Precio mediano: ${df_clean['SalePrice'].median():,.0f}")
print(f"Precio promedio: ${df_clean['SalePrice'].mean():,.0f}")
print(f"Skewness: {df_clean['SalePrice'].skew():.2f}")

### 3.2 Correlación con variables numéricas

In [ ]:
# Top correlaciones con SalePrice
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
corr_matrix = df_clean[numeric_cols].corr()
top_corr = corr_matrix['SalePrice'].drop('SalePrice').sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#2ecc71' if x > 0 else '#e74c3c' for x in top_corr.head(20)]
top_corr.head(20).plot(kind='barh', ax=ax, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Top 20 Variables más Correlacionadas con el Precio', fontweight='bold', fontsize=13)
ax.set_xlabel('Correlación de Pearson')
plt.tight_layout()
plt.savefig('../reports/figures/top_correlations.png', dpi=150)
plt.show()

### 3.3 Heatmap de correlaciones principales

In [ ]:
top_features = top_corr.head(10).index.tolist() + ['SalePrice']

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(df_clean[top_features].corr(), dtype=bool))
sns.heatmap(
    df_clean[top_features].corr(),
    annot=True, fmt='.2f', cmap='RdYlGn',
    mask=mask, ax=ax,
    linewidths=0.5, cbar_kws={'shrink': 0.8}
)
ax.set_title('Mapa de Correlaciones — Top Variables', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('../reports/figures/heatmap_correlations.png', dpi=150)
plt.show()

### 3.4 Precio por vecindario

In [ ]:
neighborhood_stats = df_clean.groupby('Neighborhood')['SalePrice'].agg(['median', 'mean', 'count'])
neighborhood_stats = neighborhood_stats.sort_values('median', ascending=True)

fig, ax = plt.subplots(figsize=(12, 9))
bars = ax.barh(neighborhood_stats.index, neighborhood_stats['median'],
               color=plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(neighborhood_stats))))
ax.axvline(df_clean['SalePrice'].median(), color='navy', linestyle='--',
           linewidth=2, label=f"Mediana global: ${df_clean['SalePrice'].median():,.0f}")
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
ax.set_title('Precio Mediano por Vecindario', fontweight='bold', fontsize=13)
ax.set_xlabel('Precio Mediano (USD)')
ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/price_by_neighborhood.png', dpi=150)
plt.show()

### 3.5 Precio vs Superficie total

In [ ]:
fig = px.scatter(
    df_clean,
    x='TotalSF', y='SalePrice',
    color='OverallQual',
    color_continuous_scale='RdYlGn',
    hover_data=['Neighborhood', 'YearBuilt', 'TotalBaths'],
    title='Precio de Venta vs Superficie Total (m²)',
    labels={'TotalSF': 'Superficie Total (sq ft)', 'SalePrice': 'Precio (USD)', 'OverallQual': 'Calidad'},
    template='plotly_white'
)
fig.update_layout(coloraxis_colorbar=dict(title='Calidad'))
fig.write_html('../reports/figures/price_vs_surface_interactive.html')
fig.show()

### 3.6 Calidad general vs Precio

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
palette = sns.color_palette('RdYlGn', n_colors=10)
sns.boxplot(data=df_clean, x='OverallQual', y='SalePrice', palette=palette, ax=ax)
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
ax.set_title('Distribución del Precio según Calidad General', fontweight='bold', fontsize=13)
ax.set_xlabel('Calidad General (1=Muy mala → 10=Excelente)')
ax.set_ylabel('Precio de Venta (USD)')
plt.tight_layout()
plt.savefig('../reports/figures/price_by_quality.png', dpi=150)
plt.show()

### 3.7 Evolución de precios por año de venta

In [ ]:
yearly = df_clean.groupby('YrSold')['SalePrice'].agg(['median', 'mean', 'count']).reset_index()

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

ax1.plot(yearly['YrSold'], yearly['median'], 'o-', color='#2ecc71', linewidth=2.5, label='Mediana')
ax1.plot(yearly['YrSold'], yearly['mean'], 's--', color='#3498db', linewidth=2, label='Promedio')
ax2.bar(yearly['YrSold'], yearly['count'], alpha=0.2, color='grey', label='Cantidad ventas')

ax1.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
ax1.set_xlabel('Año de Venta')
ax1.set_ylabel('Precio (USD)', color='#2ecc71')
ax2.set_ylabel('Cantidad de Ventas', color='grey')
ax1.set_title('Evolución del Precio Mediano por Año de Venta', fontweight='bold', fontsize=13)
ax1.legend(loc='upper left')
plt.tight_layout()
plt.savefig('../reports/figures/price_evolution.png', dpi=150)
plt.show()

### 3.8 Outliers en variables clave

In [ ]:
key_vars = ['TotalSF', 'GrLivArea', 'LotArea', 'SalePrice']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, var in enumerate(key_vars):
    sns.boxplot(y=df_clean[var], ax=axes[i], color='#3498db')
    axes[i].set_title(f'Outliers: {var}', fontweight='bold')
    Q1 = df_clean[var].quantile(0.25)
    Q3 = df_clean[var].quantile(0.75)
    IQR = Q3 - Q1
    outliers = ((df_clean[var] < Q1 - 1.5 * IQR) | (df_clean[var] > Q3 + 1.5 * IQR)).sum()
    axes[i].set_xlabel(f'{outliers} outliers detectados')

plt.suptitle('Detección de Outliers en Variables Clave', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/outliers.png', dpi=150)
plt.show()

---
## 4. Machine Learning — Modelos Predictivos

### 4.1 Preparación de datos para ML

In [ ]:
df_ml = df_clean.copy()

# Eliminar columnas con alta cardinalidad o poco valor predictivo
drop_cols = ['Id', 'YearBuilt', 'YearRemodAdd', 'GarageYrBlt']
df_ml = df_ml.drop(columns=[c for c in drop_cols if c in df_ml.columns])

# Encode variables categóricas
cat_cols = df_ml.select_dtypes(include=['object']).columns.tolist()
le = LabelEncoder()
for col in cat_cols:
    df_ml[col] = le.fit_transform(df_ml[col].astype(str))

# Variable objetivo: log-transformada para mejor rendimiento
X = df_ml.drop(columns=['SalePrice'])
y = np.log1p(df_ml['SalePrice'])

# Split train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Train: {X_train.shape} | Test: {X_test.shape}')

### 4.2 Entrenamiento y comparación de modelos

In [ ]:
models = {
    'Regresión Lineal': LinearRegression(),
    'Ridge': Ridge(alpha=10),
    'Lasso': Lasso(alpha=0.001),
    'Random Forest': RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, random_state=42)
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    cv_scores = cross_val_score(model, X, y, cv=5, scoring='neg_root_mean_squared_error')
    
    results.append({
        'Modelo': name,
        'RMSE (log)': round(rmse, 4),
        'R² Score': round(r2, 4),
        'CV RMSE (media)': round(-cv_scores.mean(), 4),
        'CV RMSE (std)': round(cv_scores.std(), 4)
    })
    print(f'✅ {name}: R²={r2:.4f} | RMSE={rmse:.4f}')

results_df = pd.DataFrame(results).sort_values('R² Score', ascending=False)
results_df

### 4.3 Comparación visual de modelos

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# R² Score
colors_r2 = sns.color_palette('RdYlGn', n_colors=len(results_df))
axes[0].barh(results_df['Modelo'], results_df['R² Score'], color=colors_r2[::-1])
axes[0].set_xlim(0, 1)
axes[0].set_title('R² Score por Modelo (mayor = mejor)', fontweight='bold')
axes[0].set_xlabel('R²')
for i, v in enumerate(results_df['R² Score']):
    axes[0].text(v + 0.01, i, f'{v:.4f}', va='center', fontsize=10)

# RMSE
colors_rmse = sns.color_palette('RdYlGn_r', n_colors=len(results_df))
axes[1].barh(results_df['Modelo'], results_df['RMSE (log)'], color=colors_rmse)
axes[1].set_title('RMSE por Modelo (menor = mejor)', fontweight='bold')
axes[1].set_xlabel('RMSE (escala log)')
for i, v in enumerate(results_df['RMSE (log)']):
    axes[1].text(v + 0.001, i, f'{v:.4f}', va='center', fontsize=10)

plt.suptitle('Comparación de Modelos Predictivos', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/model_comparison.png', dpi=150)
plt.show()

### 4.4 Importancia de variables (mejor modelo)

In [ ]:
best_model = models['Gradient Boosting']
feature_imp = pd.Series(best_model.feature_importances_, index=X.columns)
feature_imp = feature_imp.sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 8))
colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(feature_imp)))
feature_imp.sort_values().plot(kind='barh', ax=ax, color=colors)
ax.set_title('Top 20 Variables más Importantes — Gradient Boosting', fontweight='bold', fontsize=13)
ax.set_xlabel('Importancia relativa')
plt.tight_layout()
plt.savefig('../reports/figures/feature_importance.png', dpi=150)
plt.show()

### 4.5 Predicciones vs valores reales

In [ ]:
y_pred_best = best_model.predict(X_test)

# Convertir de log a USD
y_test_usd = np.expm1(y_test)
y_pred_usd = np.expm1(y_pred_best)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Scatter predicción vs real
axes[0].scatter(y_test_usd, y_pred_usd, alpha=0.5, color='#3498db', edgecolors='white', linewidth=0.5)
min_val = min(y_test_usd.min(), y_pred_usd.min())
max_val = max(y_test_usd.max(), y_pred_usd.max())
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Predicción perfecta')
axes[0].set_xlabel('Precio Real (USD)')
axes[0].set_ylabel('Precio Predicho (USD)')
axes[0].set_title('Predicción vs Real', fontweight='bold')
axes[0].xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
axes[0].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
axes[0].legend()

# Residuos
residuals = y_test_usd - y_pred_usd
axes[1].scatter(y_pred_usd, residuals, alpha=0.5, color='#e74c3c', edgecolors='white', linewidth=0.5)
axes[1].axhline(0, color='black', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Precio Predicho (USD)')
axes[1].set_ylabel('Residual (USD)')
axes[1].set_title('Gráfico de Residuos', fontweight='bold')
axes[1].xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
axes[1].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

plt.suptitle('Evaluación del Modelo — Gradient Boosting', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/predictions_vs_real.png', dpi=150)
plt.show()

rmse_usd = np.sqrt(mean_squared_error(y_test_usd, y_pred_usd))
print(f'\n📊 RMSE en USD: ${rmse_usd:,.0f}')
print(f'📊 R² Score: {r2_score(y_test_usd, y_pred_usd):.4f}')

---
## 5. Conclusiones

### 🔍 Principales hallazgos del EDA

1. **Calidad general (`OverallQual`) es el factor más determinante** del precio, con una correlación de ~0.79.
2. **La superficie total** (sótano + planta baja + alta) tiene alta correlación con el precio (>0.75).
3. **El vecindario importa significativamente**: `NoRidge`, `NridgHt` y `StoneBr` tienen precios medianos >3x superiores a los más económicos.
4. La distribución del precio tiene **alta asimetría positiva** (skewness ~1.88), lo que justifica la transformación logarítmica para ML.
5. Aproximadamente **58 columnas** tenían valores nulos, mayormente representando "ausencia de característica" (sin garage, sin pileta, etc.).

### 🤖 Resultados de Machine Learning

- El modelo **Gradient Boosting** obtuvo el mejor desempeño con R² ~0.91 en test.
- Las variables más importantes fueron: `OverallQual`, `TotalSF`, `GrLivArea`, `GarageCars`, `TotalBaths`.
- La transformación log del target mejoró significativamente el rendimiento de todos los modelos.

### 💡 Insights de negocio

- Invertir en mejorar la calidad general de la construcción tiene el mayor impacto en el precio de venta.
- Una casa con garage puede valer significativamente más que una equivalente sin él.
- Casas remodeladas recientemente tienden a venderse por encima del valor de mercado.
